# 05 - Extended Gold Products and Agent/Ontology Context

Builds persona-ready operational, spatial, reliability, efficiency, executive, and IT serving products. It then replaces the legacy `agent_context` with a backward-compatible superset at the same one-row-per-gate grain.

All recommendations are synthetic and advisory; consequential recommendations require human approval.

In [ ]:
from pyspark.sql import functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
observation_ts = config['observation_timestamp'].strftime('%Y-%m-%d %H:%M:%S')

def build(sql_text, table_name):
    spark.sql(sql_text.replace('__OBSERVATION_TS__', observation_ts))
    print(table_name, spark.table(table_name).count())

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_airport_operational_health AS
WITH asset AS (
  SELECT airport_id, ROUND(AVG(availability_pct), 1) AS asset_availability_pct,
         SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS asset_anomaly_observations
  FROM fact_asset_state GROUP BY airport_id),
incidents AS (
  SELECT airport_id, SUM(CASE WHEN severity = 'High' AND status = 'Open' THEN 1 ELSE 0 END) AS open_high_incidents
  FROM fact_operational_incidents GROUP BY airport_id)
SELECT k.airport_id, k.flights, k.on_time_departure_rate, k.avg_turnaround_min,
       k.avg_queue_wait_min, k.energy_kwh_per_flight, k.energy_kwh_per_pax,
       k.maintenance_anomaly_count, k.incident_count,
       a.asset_availability_pct, a.asset_anomaly_observations,
       COALESCE(i.open_high_incidents, 0) AS open_high_incidents,
       ROUND(LEAST(100.0, (100.0 - k.on_time_departure_rate) * 0.45 +
             k.avg_queue_wait_min * 1.5 + k.maintenance_anomaly_count * 2.0 +
             COALESCE(i.open_high_incidents, 0) * 8.0 + (100.0 - COALESCE(a.asset_availability_pct, 100.0))), 1) AS operational_risk_score,
       CASE WHEN (100.0 - k.on_time_departure_rate) + COALESCE(i.open_high_incidents, 0) * 10 >= 35 THEN 'High'
            WHEN (100.0 - k.on_time_departure_rate) + COALESCE(i.open_high_incidents, 0) * 10 >= 18 THEN 'Medium'
            ELSE 'Low' END AS risk_category,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM gold_kpi_daily_summary k
LEFT JOIN asset a ON k.airport_id = a.airport_id
LEFT JOIN incidents i ON k.airport_id = i.airport_id
""", 'gold_airport_operational_health')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_terminal_flow_summary AS
SELECT z.airport_id, z.terminal_id, o.date_key, o.event_hour,
       ROUND(AVG(o.occupancy_count), 1) AS avg_zone_occupancy,
       ROUND(AVG(o.wait_time_min), 1) AS avg_queue_wait_min,
       MAX(o.wait_time_min) AS peak_queue_wait_min,
       SUM(o.throughput_pax) AS passenger_throughput,
       COUNT(DISTINCT o.checkpoint_id) AS observed_checkpoints,
       CASE WHEN MAX(o.wait_time_min) >= 25 THEN 'Constrained'
            WHEN MAX(o.wait_time_min) >= 15 THEN 'Watch' ELSE 'Normal' END AS flow_status,
       true AS is_synthetic
FROM fact_zone_occupancy o
JOIN dim_zone z ON o.zone_id = z.zone_id
GROUP BY z.airport_id, z.terminal_id, o.date_key, o.event_hour
""", 'gold_terminal_flow_summary')

build("""
CREATE OR REPLACE TABLE gold_gate_turnaround_performance AS
SELECT g.airport_id, t.terminal_id, g.gate_id, b.stand_id,
       COUNT(*) AS flights, ROUND(AVG(f.turnaround_minutes), 1) AS avg_turnaround_min,
       ROUND(AVG(a.turnaround_target_min), 1) AS avg_turnaround_target_min,
       ROUND(AVG(CASE WHEN f.turnaround_minutes <= a.turnaround_target_min THEN 1.0 ELSE 0.0 END) * 100, 1) AS target_adherence_pct,
       ROUND(AVG(f.on_time_flag) * 100, 1) AS on_time_departure_rate,
       ROUND(MAX(f.departure_delay_minutes), 1) AS max_departure_delay_min,
       MAX_BY(f.delay_reason, f.departure_delay_minutes) AS primary_delay_reason,
       u.utilization_pct, true AS is_synthetic
FROM fact_flight_turnaround_events f
JOIN dim_gate g ON f.gate_id = g.gate_id
JOIN dim_terminal t ON g.airport_id = t.airport_id AND g.terminal = t.terminal_code
JOIN dim_aircraft a ON f.aircraft_type_id = a.aircraft_type_id
LEFT JOIN bridge_gate_stand b ON g.gate_id = b.gate_id AND b.is_current = true
LEFT JOIN gold_gate_utilization u ON g.gate_id = u.gate_id
GROUP BY g.airport_id, t.terminal_id, g.gate_id, b.stand_id, u.utilization_pct
""", 'gold_gate_turnaround_performance')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_asset_reliability AS
WITH state AS (
  SELECT asset_id, ROUND(AVG(availability_pct), 1) AS availability_pct,
         SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS anomaly_count,
         MAX(event_time) AS latest_telemetry_timestamp,
         MAX_BY(health_status, event_time) AS latest_health_status
  FROM fact_asset_state GROUP BY asset_id),
maintenance AS (
  SELECT gate_id, asset_type, COUNT(*) AS maintenance_event_count,
         SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS maintenance_anomaly_count,
         SUM(CASE WHEN status <> 'Closed' THEN 1 ELSE 0 END) AS open_maintenance_count
  FROM fact_maintenance_events GROUP BY gate_id, asset_type)
SELECT a.airport_id, a.terminal_id, a.zone_id, a.gate_id, a.asset_id, a.asset_type,
       a.asset_class, a.criticality, s.availability_pct, s.anomaly_count,
       COALESCE(m.maintenance_event_count, 0) AS maintenance_event_count,
       COALESCE(m.maintenance_anomaly_count, 0) AS maintenance_anomaly_count,
       COALESCE(m.open_maintenance_count, 0) AS open_maintenance_count,
       s.latest_telemetry_timestamp, s.latest_health_status,
       CASE WHEN s.anomaly_count > 0 OR COALESCE(m.maintenance_anomaly_count, 0) > 0 THEN 'Anomalous'
            WHEN s.availability_pct < 97 OR COALESCE(m.open_maintenance_count, 0) > 0 THEN 'Degraded'
            ELSE 'Healthy' END AS reliability_status,
       true AS is_synthetic
FROM dim_asset a
JOIN state s ON a.asset_id = s.asset_id
LEFT JOIN maintenance m ON a.gate_id = m.gate_id AND a.asset_type = m.asset_type
""", 'gold_asset_reliability')

build("""
CREATE OR REPLACE TABLE gold_energy_efficiency AS
WITH energy AS (SELECT airport_id, gate_id, SUM(kwh) AS total_kwh FROM fact_energy_metering GROUP BY airport_id, gate_id),
flights AS (SELECT airport_id, gate_id, COUNT(*) AS flights, SUM(passenger_count) AS passengers FROM fact_flight_turnaround_events GROUP BY airport_id, gate_id)
SELECT e.airport_id, t.terminal_id, e.gate_id, f.flights, f.passengers, ROUND(e.total_kwh, 1) AS total_kwh,
       ROUND(e.total_kwh / f.flights, 1) AS energy_kwh_per_flight,
       ROUND(e.total_kwh / f.passengers, 3) AS energy_kwh_per_passenger,
       CASE WHEN e.total_kwh / f.flights > 550 THEN 'High'
            WHEN e.total_kwh / f.flights > 450 THEN 'Watch' ELSE 'Efficient' END AS efficiency_status,
       true AS is_synthetic
FROM energy e JOIN flights f ON e.airport_id = f.airport_id AND e.gate_id = f.gate_id
JOIN dim_gate g ON e.gate_id = g.gate_id
JOIN dim_terminal t ON g.airport_id = t.airport_id AND g.terminal = t.terminal_code
""", 'gold_energy_efficiency')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_spatial_operational_status AS
WITH flow AS (
  SELECT zone_id, ROUND(AVG(wait_time_min), 1) AS avg_queue_wait_min, MAX(wait_time_min) AS peak_queue_wait_min,
         ROUND(AVG(occupancy_count), 1) AS avg_occupancy
  FROM fact_zone_occupancy GROUP BY zone_id),
asset AS (
  SELECT zone_id, COUNT(*) AS asset_count, SUM(CASE WHEN reliability_status = 'Anomalous' THEN 1 ELSE 0 END) AS anomalous_asset_count
  FROM gold_asset_reliability GROUP BY zone_id)
SELECT z.airport_id, z.terminal_id, z.zone_id, l.location_id, l.spatial_ref,
       COALESCE(f.avg_queue_wait_min, 0.0) AS avg_queue_wait_min,
       COALESCE(f.peak_queue_wait_min, 0.0) AS peak_queue_wait_min,
       COALESCE(f.avg_occupancy, 0.0) AS avg_occupancy,
       COALESCE(a.asset_count, 0) AS asset_count, COALESCE(a.anomalous_asset_count, 0) AS anomalous_asset_count,
       CASE WHEN COALESCE(a.anomalous_asset_count, 0) > 0 OR COALESCE(f.peak_queue_wait_min, 0) >= 25 THEN 'Action'
            WHEN COALESCE(f.peak_queue_wait_min, 0) >= 15 THEN 'Watch' ELSE 'Normal' END AS operational_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp, true AS is_synthetic
FROM dim_zone z
JOIN dim_location l ON l.location_id = CONCAT('LOC-', z.zone_id)
LEFT JOIN flow f ON z.zone_id = f.zone_id
LEFT JOIN asset a ON z.zone_id = a.zone_id
""", 'gold_spatial_operational_status')

build("""
CREATE OR REPLACE TABLE gold_executive_scorecard AS
SELECT h.airport_id, a.airport_name, h.operational_risk_score, h.risk_category,
       h.on_time_departure_rate, h.avg_turnaround_min, h.avg_queue_wait_min,
       h.asset_availability_pct, h.energy_kwh_per_flight, h.energy_kwh_per_pax,
       h.open_high_incidents, h.maintenance_anomaly_count, h.observation_timestamp,
       CASE WHEN h.risk_category = 'High' THEN 'Executive attention required'
            WHEN h.risk_category = 'Medium' THEN 'Monitor operational exceptions'
            ELSE 'Operating within demo thresholds' END AS executive_commentary,
       true AS is_synthetic
FROM gold_airport_operational_health h JOIN dim_airport a ON h.airport_id = a.airport_id
""", 'gold_executive_scorecard')

In [ ]:
# Backward-compatible agent_context: legacy columns remain; new fields make grounding explicit.
build("""
CREATE OR REPLACE TABLE agent_context AS
WITH ranked_asset AS (
  SELECT r.*, ROW_NUMBER() OVER (PARTITION BY gate_id ORDER BY anomaly_count DESC, maintenance_anomaly_count DESC, availability_pct ASC, asset_id) AS rn
  FROM gold_asset_reliability r),
ranked_incident AS (
  SELECT i.*, ROW_NUMBER() OVER (PARTITION BY gate_id ORDER BY
    CASE severity WHEN 'High' THEN 3 WHEN 'Medium' THEN 2 ELSE 1 END DESC, delay_minutes DESC, incident_id) AS rn
  FROM fact_operational_incidents i),
terminal_flow AS (
  SELECT airport_id, terminal_id, ROUND(AVG(avg_queue_wait_min), 1) AS avg_queue_wait_min
  FROM gold_terminal_flow_summary GROUP BY airport_id, terminal_id)
SELECT p.airport_id, p.gate_id,
       CASE WHEN p.max_departure_delay_min > 30 OR i.severity = 'High' OR a.reliability_status = 'Anomalous' THEN 'Action'
            WHEN p.max_departure_delay_min > 15 OR tf.avg_queue_wait_min >= 15 OR a.reliability_status = 'Degraded' THEN 'Watch'
            ELSE 'Normal' END AS operational_status,
       COALESCE(NULLIF(p.primary_delay_reason, 'None'), i.category, 'None') AS delay_reason,
       CASE WHEN i.severity = 'High' THEN 'Duty manager to review the incident and approve a response plan'
            WHEN a.reliability_status = 'Anomalous' THEN 'Maintenance lead to inspect the asset and approve any intervention'
            WHEN p.max_departure_delay_min > 15 THEN 'Operations lead to review gate resourcing and approve adjustments'
            ELSE 'Continue monitoring; no consequential action recommended' END AS recommended_action,
       p.terminal_id, a.zone_id, p.stand_id, a.asset_id,
       p.on_time_departure_rate, p.avg_turnaround_min, p.avg_turnaround_target_min,
       p.target_adherence_pct, p.utilization_pct AS gate_utilization_pct, tf.avg_queue_wait_min,
       a.availability_pct AS asset_availability_pct, a.anomaly_count AS asset_anomaly_count,
       i.incident_id AS relevant_incident_id, i.category AS relevant_incident_category,
       l.location_id AS spatial_location_id, l.spatial_ref,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       CASE WHEN i.severity = 'High' THEN 'A high-severity synthetic incident affects this gate'
            WHEN a.reliability_status = 'Anomalous' THEN 'Deterministic telemetry indicates an anomalous asset state'
            WHEN p.max_departure_delay_min > 15 THEN 'Observed departure delay exceeds the 15-minute on-time threshold'
            ELSE 'KPIs are within the configured synthetic thresholds' END AS recommendation_rationale,
       CASE WHEN i.severity = 'High' OR p.max_departure_delay_min > 30 THEN 'High'
            WHEN a.reliability_status <> 'Healthy' OR p.max_departure_delay_min > 15 THEN 'Medium' ELSE 'Informational' END AS severity,
       CASE WHEN a.asset_id IS NOT NULL AND p.stand_id IS NOT NULL AND l.spatial_ref IS NOT NULL THEN 'High' ELSE 'Medium' END AS confidence_category,
       CONCAT('gold_gate_turnaround_performance;gold_asset_reliability;gold_terminal_flow_summary;',
              CASE WHEN i.incident_id IS NULL THEN '' ELSE 'fact_operational_incidents;' END, 'dim_location') AS source_table_references,
       CASE WHEN i.severity = 'High' OR a.reliability_status = 'Anomalous' OR p.max_departure_delay_min > 15 THEN true ELSE false END AS human_approval_required,
       CASE WHEN a.latest_telemetry_timestamp >= CAST('__OBSERVATION_TS__' AS TIMESTAMP) - INTERVAL 6 HOURS THEN 'CurrentWithin6Hours' ELSE 'DelayedOver6Hours' END AS data_freshness_indicator,
       true AS advisory_only, true AS is_synthetic
FROM gold_gate_turnaround_performance p
JOIN ranked_asset a ON p.gate_id = a.gate_id AND a.rn = 1
LEFT JOIN ranked_incident i ON p.gate_id = i.gate_id AND i.rn = 1
LEFT JOIN terminal_flow tf ON p.airport_id = tf.airport_id AND p.terminal_id = tf.terminal_id
LEFT JOIN dim_location l ON l.location_id = CONCAT('LOC-', a.asset_id)
""", 'agent_context')

In [ ]:
# Synthetic IT/service health. Capacity and cost fields are explicit proxies, not tenant telemetry.
airport_count = int(config['airport_count'])
gate_count = airport_count * int(config['gates_per_airport'])
zone_count = airport_count * int(config['terminals_per_airport']) * int(config['zones_per_terminal'])
queue_count = airport_count * int(config['checkpoints_per_airport']) * (int(config['simulation_hours']) * 60 // int(config['queue_interval_minutes']))
asset_count = gate_count * 6
asset_state_count = asset_count * (int(config['simulation_hours']) // int(config['asset_state_interval_hours']))
flight_count = gate_count * int(config['flights_per_gate'])
products = [
    ('Bronze Flight Events','Bronze','bronze_flight_turnaround',flight_count),
    ('Silver Turnaround','Silver','fact_flight_turnaround_events',flight_count),
    ('Silver Zone Occupancy','Silver','fact_zone_occupancy',queue_count),
    ('Silver Asset State','Silver','fact_asset_state',asset_state_count),
    ('Gold Executive Scorecard','Gold','gold_executive_scorecard',airport_count),
    ('Gold Gate Performance','Gold','gold_gate_turnaround_performance',gate_count),
    ('Gold Spatial Status','Gold','gold_spatial_operational_status',zone_count),
    ('Agent Grounding Context','Gold','agent_context',gate_count)]
health_rows = []
for index, (product, layer, table_name, expected_rows) in enumerate(products):
    actual_rows = spark.table(table_name).count()
    health_rows.append((
        product, layer, table_name, expected_rows, actual_rows,
        100.0 if actual_rows == expected_rows else 0.0, 0, 0,
        'SyntheticSuccess' if actual_rows == expected_rows else 'SyntheticFailure',
        'CurrentAsOfDemoObservation', 55.0 + index * 3.0,
        100.0 + index * 7.5, 'SyntheticControlPassed',
        config['observation_timestamp'], 'Derived analytical data', True))
health_schema = 'data_product string, layer string, source_table string, expected_row_count long, actual_row_count long, data_quality_pass_pct double, late_record_count long, quarantined_record_count long, pipeline_run_status string, refresh_status string, synthetic_capacity_usage_pct double, synthetic_cost_proxy_units double, security_control_status string, observation_timestamp timestamp, data_classification string, is_synthetic boolean'
health_df = spark.createDataFrame(health_rows, health_schema)
health_df.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable('gold_it_service_health')
print('gold_it_service_health', health_df.count())

In [ ]:
# Mandatory configuration-derived Gold contract checks.
terminal_hour_count = airport_count * int(config['terminals_per_airport']) * int(config['simulation_hours'])
required_gold = {
    'gold_airport_operational_health': airport_count,
    'gold_terminal_flow_summary': terminal_hour_count,
    'gold_gate_turnaround_performance': gate_count,
    'gold_asset_reliability': asset_count,
    'gold_energy_efficiency': gate_count,
    'gold_spatial_operational_status': zone_count,
    'gold_executive_scorecard': airport_count,
    'gold_it_service_health': len(products),
    'agent_context': gate_count}
for table_name, expected_count in required_gold.items():
    actual_count = spark.table(table_name).count()
    assert actual_count == expected_count, table_name + ' expected ' + str(expected_count) + ' rows, got ' + str(actual_count)
legacy_columns = {'airport_id','gate_id','operational_status','delay_reason','recommended_action'}
assert legacy_columns.issubset(set(spark.table('agent_context').columns))
assert spark.table('agent_context').filter(~F.col('advisory_only') | ~F.col('is_synthetic')).count() == 0
assert spark.table('agent_context').filter(F.col('source_table_references').isNull()).count() == 0
print('PASS: configuration-derived Gold and backward-compatible agent_context checks')